Wednesday: the tie decides the day

> "Retail-Plus frequency is the problem, so we want to protect our best members before they
> drift. Give us the top fifty customers by Q2 revenue in each segment, and flag anyone whose
> monthly spend has fallen for two months running."

And from the head of Retail-Plus, who owns the tier:

> "Ties matter. If two members spent the same, I want them ranked the same, and I want to know
> how many made the top fifty, not forty-nine because of a tie."

**MAP** collapse against keep  ->  **DO** rank  ->  **SEE** the tie  ->
**CHECK** the flag  ->  **SUM** what you hand Marketing.

In [1]:
import pathlib
import sys

root = next(p for p in pathlib.Path.cwd().resolve().parents if (p / "scripts" / "c2kit.py").exists())
sys.path.insert(0, str(root / "scripts"))
import c2kit as kit

conn = kit.connect()
n = kit.sql("""SELECT count(DISTINCT o.customer_id) AS members FROM orders o
               JOIN customers c USING (customer_id)
               WHERE o.quarter='Q2' AND c.segment='Retail-Plus'""", conn=conn)[0]
print(f"{n['members']} Retail-Plus members ordered in Q2, and Marketing wants fifty of them")

76 Retail-Plus members ordered in Q2, and Marketing wants fifty of them


## MAP. Two shapes of question

`GROUP BY` answers how much per group, and the rows collapse. Marketing's questions keep the
rows and ask about each row's neighbours: its rank within its segment, its own previous month,
the total so far.

In [2]:
kit.matrix(["GROUP BY", "window"],
           ["what happens to the rows", "the question it answers"],
           [["they collapse", "how much per group"],
            ["they survive", "where does this row stand"]],
           title="Why a window exists at all")

## DO. Rank the members, and look at the boundary

Nothing collapses. Each row gains a fact about where it sits among its neighbours. Look at
positions 46 to 54 rather than the top ten, because the top of a list is never where the
argument happens.

In [3]:
kit.sql_table("""
    WITH q2 AS (SELECT o.customer_id, sum(o.amount) AS revenue
                FROM orders o JOIN customers c USING (customer_id)
                WHERE o.quarter='Q2' AND c.segment='Retail-Plus'
                GROUP BY o.customer_id),
    r AS (SELECT customer_id, revenue,
                 row_number() OVER (ORDER BY revenue DESC) AS rn,
                 rank()       OVER (ORDER BY revenue DESC) AS rk,
                 dense_rank() OVER (ORDER BY revenue DESC) AS dr
          FROM q2)
    SELECT rn, rk, dr, revenue FROM r WHERE rn BETWEEN 46 AND 54 ORDER BY rn
""", caption="Three functions, and two ties in nine rows", conn=conn)

rn,rk,dr,revenue
46,46,46,3600
47,47,47,3540
48,48,48,3480
49,48,48,3480
50,50,49,3350
51,50,49,3350
52,52,50,3200
53,53,51,3150
54,54,52,3110


### Two rows share a revenue, and they sit at fifty and fifty-one

Nobody engineered that for a slide. It is what happens when fifty is a round number somebody
picked and revenue is measured in rupees.

In [4]:
counts = kit.sql("""
    WITH q2 AS (SELECT o.customer_id, sum(o.amount) AS revenue
                FROM orders o JOIN customers c USING (customer_id)
                WHERE o.quarter='Q2' AND c.segment='Retail-Plus' GROUP BY o.customer_id),
    r AS (SELECT row_number() OVER (ORDER BY revenue DESC) rn,
                 rank()       OVER (ORDER BY revenue DESC) rk,
                 dense_rank() OVER (ORDER BY revenue DESC) dr FROM q2)
    SELECT count(*) FILTER (WHERE rn<=50) AS by_row_number,
           count(*) FILTER (WHERE rk<=50) AS by_rank,
           count(*) FILTER (WHERE dr<=50) AS by_dense_rank FROM r
""", conn=conn)[0]
for k, v in counts.items():
    print(f"  top fifty by {k.replace('by_',''):<12} ships {v} names")
kit.check("the three rules disagree about the size of a top fifty",
          len(set(counts.values())) == 3, str(dict(counts)))
kit.check("RANK is the rule that ships fifty-one", counts["by_rank"] == 51)

  top fifty by row_number   ships 50 names
  top fifty by rank         ships 51 names
  top fifty by dense_rank   ships 52 names


### The head of Retail-Plus already answered this

"Ranked the same" rules out `ROW_NUMBER`, which breaks a tie arbitrarily and is not even
repeatable between runs. "Not forty-nine because of a tie" rules out cutting the tie off.
`RANK` at fifty ships fifty-one names, and the extra name is the point.

Put that sentence in the query as a comment. The next person will not have heard him say it.

In [5]:
kit.decision_ladder(["ROW_NUMBER: 50 names, one of the tied pair dropped arbitrarily",
                     "DENSE_RANK: 52 names, the cut moves further down",
                     "RANK: 51 names, both tied members kept"],
                    cut_at=2, title="What the business asked for")

## SEE. The error that sends you back to Monday's picture

A window is computed after `WHERE` has already run, so filtering on one directly is refused.
This is the same execution order that refused an aggregate in `WHERE` on Monday.

In [6]:
try:
    kit.sql("SELECT customer_id FROM orders WHERE rank() OVER (ORDER BY amount) <= 5", conn=conn)
except Exception as e:
    conn.rollback()
    print(str(e).strip().splitlines()[0])
    print("\nCompute the window in an inner query or a CTE, then filter outside it.")

wrapped = kit.sql("""
    WITH q2 AS (SELECT c.segment, o.customer_id, sum(o.amount) AS revenue
                FROM orders o JOIN customers c USING (customer_id)
                WHERE o.quarter='Q2' GROUP BY c.segment, o.customer_id),
    r AS (SELECT segment, customer_id, revenue,
                 rank() OVER (PARTITION BY segment ORDER BY revenue DESC) AS pos FROM q2)
    SELECT segment, count(*) AS names FROM r WHERE pos <= 50 GROUP BY segment ORDER BY segment
""", conn=conn)
kit.table(["segment", "names in the top fifty"], [[r["segment"], r["names"]] for r in wrapped],
          caption="PARTITION BY restarts the numbering for each segment")
kit.check("Retail-Plus ships fifty-one names",
          next(r["names"] for r in wrapped if r["segment"] == "Retail-Plus") == 51)

window functions are not allowed in WHERE

Compute the window in an inner query or a CTE, then filter outside it.


segment,names in the top fifty
Business,35
Retail-Core,50
Retail-Plus,51
Student,20


## CHECK. Falling for two months running

`LAG` reads the previous row within the same customer, in month order. Two LAGs and a comparison
answers Marketing's second ask.

In [7]:
kit.vflow(["July", "August", "September"], lit=[2], title="lag(spend, 1) and lag(spend, 2)")
falling = kit.sql("""
    WITH monthly AS (
        SELECT o.customer_id, date_trunc('month', o.order_date) AS mth, sum(o.amount) AS spend
        FROM orders o JOIN customers c USING (customer_id)
        WHERE o.quarter='Q2' AND c.segment='Retail-Plus'
        GROUP BY o.customer_id, date_trunc('month', o.order_date)),
    l AS (SELECT customer_id, mth, spend,
                 lag(spend,1) OVER (PARTITION BY customer_id ORDER BY mth) AS m1,
                 lag(spend,2) OVER (PARTITION BY customer_id ORDER BY mth) AS m2
          FROM monthly)
    SELECT customer_id, m2 AS july, m1 AS august, spend AS september
    FROM l WHERE m2 > m1 AND m1 > spend ORDER BY customer_id
""", conn=conn)
kit.table(["member", "July", "August", "September"],
          [[r["customer_id"], kit.rupees(r["july"]), kit.rupees(r["august"]),
            kit.rupees(r["september"])] for r in falling],
          caption="Three members, three months, falling each time")
kit.check("three members fall in both steps", len(falling) == 3, f"{len(falling)}")
kit.check("each one really is monotone", all(r["july"] > r["august"] > r["september"]
                                             for r in falling))

member,July,August,September
C-0161,"Rs 4,200","Rs 3,100","Rs 1,900"
C-0171,"Rs 3,800","Rs 2,600","Rs 1,400"
C-0175,"Rs 4,400","Rs 2,900","Rs 1,600"


### The first month of every customer returns NULL

There is no previous row, so `lag` has nothing to give. That is correct, and it will quietly
drop customers from the flag if the comparison is written without thinking about it.

A member with one month of data is not falling and is not steady. You cannot tell, and the
honest flag says so rather than guessing.

## SUM. The running total, and what makes it reproducible

Revenue accumulates without the weeks collapsing. The window's `ORDER BY` is what makes the
cumulative column deterministic: if two rows can share a position, their order is undefined and
the number can differ between runs.

In [8]:
plan = kit.sql("""
    WITH weekly AS (SELECT date_trunc('week', order_date) AS week_start, sum(amount) AS revenue
                    FROM orders WHERE quarter='Q2' GROUP BY date_trunc('week', order_date))
    SELECT w.week_start::date AS week, w.revenue,
           sum(w.revenue) OVER (ORDER BY w.week_start) AS cumulative,
           sum(p.plan_revenue) OVER (ORDER BY w.week_start) AS plan_cumulative
    FROM weekly w LEFT JOIN plan_line p ON p.week_start = w.week_start::date
    ORDER BY w.week_start
""", conn=conn)
mid = plan[6]
print(f"by {mid['week']}, actual {kit.rupees(mid['cumulative'])} "
      f"against plan {kit.rupees(mid['plan_cumulative'])}")
last = plan[-1]
print(f"by {last['week']}, actual {kit.rupees(last['cumulative'])} "
      f"against plan {kit.rupees(last['plan_cumulative'])}")
kit.check("the quarter runs ahead of plan at the halfway mark",
          mid["cumulative"] > mid["plan_cumulative"])
kit.check("and lands level with it", abs(float(last["cumulative"]) -
                                         float(last["plan_cumulative"])) < 1000)

by 2026-08-10, actual Rs 6,32,84,780 against plan Rs 4,54,15,380
by 2026-09-28, actual Rs 9,84,00,000 against plan Rs 9,83,99,990


### That shape is the finding, not the endpoint

Revenue is ahead of plan for most of the quarter and then the weekly run rate falls away, so it
only just lands. A report that shows the final number alone says Kalpa hit plan. The running
total says it hit plan while slowing down, which is a different conversation.

In [9]:
kit.ladder(["ahead by mid-quarter", "run rate falls", "lands level", "the slowdown is the story"],
           lit=[3], title="What the accumulation shows that the total hides")
kit.check_summary()